# Phase 10 Training Analysis
Inspect PPO training outcomes, baseline-beat status, and evaluation comparison against Phase 9 baselines.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

phase10_root = Path('Simulation_4/artifacts/phase10')
summary_path = phase10_root / 'training_summary.json'
log_path = phase10_root / 'training_log.json'
eval_path = phase10_root / 'evaluation_report.json'

print('Summary exists:', summary_path.exists())
print('Training log exists:', log_path.exists())
print('Eval report exists:', eval_path.exists())

Summary exists: False
Training log exists: False
Eval report exists: False


In [2]:
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(summary, indent=2))
else:
    summary = {}
    print('Run training first: python Simulation_4/scripts/phase10_train.py --timesteps 500000')

Run training first: python Simulation_4/scripts/phase10_train.py --timesteps 500000


In [3]:
if log_path.exists():
    train_log = json.loads(log_path.read_text(encoding='utf-8'))
    metrics_df = pd.DataFrame(train_log.get('metrics_history', []))
    eval_df = pd.DataFrame(train_log.get('eval_history', []))

    fig, ax = plt.subplots(figsize=(10, 4))
    if not metrics_df.empty:
        ax.plot(metrics_df['timesteps'], metrics_df['mean_reward_100ep'], label='Train mean_reward_100ep')
    if not eval_df.empty:
        ax.plot(eval_df['timesteps'], eval_df['mean_reward'], marker='o', label='Eval mean_reward')
    ax.axhline(0.99, linestyle='--', color='red', label='document_guided baseline = 0.99')
    ax.set_title('Phase 10 PPO Reward Curves')
    ax.set_xlabel('Timesteps')
    ax.set_ylabel('Reward')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.show()

    display(metrics_df.tail(10))
    display(eval_df.tail(10))
else:
    print('training_log.json not found yet.')

training_log.json not found yet.


In [4]:
if eval_path.exists():
    report = json.loads(eval_path.read_text(encoding='utf-8'))
    ppo = report.get('ppo', {})
    base = report.get('baselines', {})

    print('PPO mean_reward:', round(float(ppo.get('mean_reward', 0.0)), 4))
    print('PPO resolution_rate:', round(float(ppo.get('resolution_rate', 0.0)), 4))
    print('PPO escalation_rate:', round(float(ppo.get('escalation_rate', 0.0)), 4))

    rows = []
    rows.append({'policy': 'ppo_trained', **{k: v for k, v in ppo.items() if isinstance(v, (int, float))}})
    for name, vals in base.items():
        rows.append({'policy': name, **{k: v for k, v in vals.items() if isinstance(v, (int, float))}})

    comp_df = pd.DataFrame(rows)
    comp_df = comp_df.sort_values('mean_reward', ascending=False)
    display(comp_df[['policy', 'mean_reward', 'resolution_rate', 'escalation_rate', 'mean_turns_to_resolution']])

    print('PPO escalation by tier:', ppo.get('escalation_by_tier', {}))
else:
    print('Run evaluation first: python Simulation_4/scripts/phase10_evaluate.py')

Run evaluation first: python Simulation_4/scripts/phase10_evaluate.py
